# 3. Bayesian Parameter Estimation

We treat the **same fitting problem** as in `01_maximum_likelihood_fitting.ipynb`: a PDF for a continuous random variable $x$ defined as the sum of a Gaussian and an exponential:

$$f(x;\theta,\mu,\sigma,\lambda) = \theta \cdot \mathcal{N}(x;\mu,\sigma) + (1-\theta) \cdot \mathcal{E}(x;\lambda)$$

where:
- $\theta \in [0,1]$ is the **parameter of interest** (signal fraction)
- $\mu,\sigma$ are the mean and width of the Gaussian (**nuisance parameters**)
- $\lambda > 0$ is the rate of the exponential (**nuisance parameter**)

### Bayesian approach

Using **Bayes' theorem**, the posterior pdf of the parameters given the data is:

$$p(\theta,\mu,\sigma,\lambda \mid \mathbf{x}) \propto \mathcal{L}(\mathbf{x};\theta,\mu,\sigma,\lambda)\cdot \pi(\theta,\mu,\sigma,\lambda)$$

where $\pi$ denotes the **prior** pdf.  The parameters are summarised using the **posterior mode** (MAP estimators).  The posterior is **marginalized** over the nuisance parameters $(\mu,\sigma,\lambda)$ using **Markov Chain Monte Carlo (MCMC)** to obtain the marginal posterior for $\theta$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon
from scipy.optimize import minimize
import emcee
import corner

rng = np.random.default_rng(seed=123)

## 1. True parameters and data generation

We use the **same true parameters and dataset** as in
[01_maximum_likelihood_fitting.ipynb](01_maximum_likelihood_fitting.ipynb)
(identical random seed → identical data, shown there). This allows direct comparison
of the Bayesian posterior with the MLE and profile-likelihood CI from notebooks 01–02.

In [ ]:
# ------- true parameter values -------
theta_true  = 0.3   # signal fraction
mu_true     = 5.0   # Gaussian mean
sigma_true  = 0.5   # Gaussian width
lambda_true = 0.5   # exponential rate  (mean = 1/lambda)

n_samples = 2000    # total number of events

# ------- sampling from the mixture -------
n_signal     = rng.binomial(n_samples, theta_true)
n_background = n_samples - n_signal

x_signal     = rng.normal(mu_true, sigma_true, size=n_signal)
# scipy's expon uses scale = 1/lambda
x_background = rng.exponential(1.0 / lambda_true, size=n_background)

x_data = np.concatenate([x_signal, x_background])
rng.shuffle(x_data)

print(f"Generated {len(x_data)} events  "
      f"({n_signal} signal + {n_background} background)")

## 2. Log-likelihood and priors

The log-likelihood is the same as in the MLE case:

$$\ln \mathcal{L} = \sum_{i=1}^{n} \ln\!\left[\theta \, \mathcal{N}(x_i;\mu,\sigma) + (1-\theta)\, \mathcal{E}(x_i;\lambda)\right]$$

We adopt **weakly informative priors** that encode only basic physical constraints:

| Parameter | Prior |
|---|---|
| $\theta$ | $\mathrm{Uniform}(0, 1)$ |
| $\mu$ | $\mathrm{Uniform}(0, 14)$ |
| $\sigma$ | $\mathrm{Uniform}(0.01, 5)$ |
| $\lambda$ | $\mathrm{Uniform}(0.01, 5)$ |

In [ ]:
def log_likelihood(params, data):
    """Unbinned log-likelihood for the Gaussian + Exponential mixture."""
    theta, mu, sigma, lam = params

    log_sig_term = np.log(theta)       + norm.logpdf(data, mu, sigma)
    log_bkg_term = np.log(1.0 - theta) + expon.logpdf(data, scale=1.0/lam)

    return np.sum(np.logaddexp(log_sig_term, log_bkg_term))


def log_prior(params):
    """Log of the joint prior (uniform in the physically allowed region)."""
    theta, mu, sigma, lam = params

    if not (0.0 < theta < 1.0):
        return -np.inf
    if not (0.0 < mu < 14.0):
        return -np.inf
    if not (0.01 < sigma < 5.0):
        return -np.inf
    if not (0.01 < lam < 5.0):
        return -np.inf

    return 0.0   # log(1) inside the allowed region


def log_posterior(params, data):
    """Log-posterior = log-likelihood + log-prior."""
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(params, data)

## 3. MAP estimator (posterior mode)

The **Maximum A Posteriori (MAP)** estimate maximises the log-posterior.  It equals the MLE when the prior is uniform, but it is the natural Bayesian point estimate and provides a good starting point for MCMC.

In [ ]:
def neg_log_posterior(params, data):
    return -log_posterior(params, data)


# Initial guess
p0_map = [0.2, x_data.mean(), 0.6, 0.6]

result_map = minimize(
    neg_log_posterior,
    x0=p0_map,
    args=(x_data,),
    method='Nelder-Mead',
    options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 50000}
)

theta_map, mu_map, sigma_map, lam_map = result_map.x

print("=== MAP estimates ===")
print(f"  θ̂_MAP = {theta_map:.4f}   (true: {theta_true})")
print(f"  μ̂_MAP = {mu_map:.4f}    (true: {mu_true})")
print(f"  σ̂_MAP = {sigma_map:.4f}   (true: {sigma_true})")
print(f"  λ̂_MAP = {lam_map:.4f}   (true: {lambda_true})")
print(f"  Optimiser success: {result_map.success}")

## 4. MCMC sampling with `emcee`

We use the **affine-invariant ensemble sampler** from `emcee` to draw samples from the posterior.  Starting walkers are initialised in a tight ball around the MAP estimate.

In [ ]:
ndim     = 4          # (theta, mu, sigma, lambda)
nwalkers = 32         # number of parallel walkers (must be > 2*ndim)
nburn    = 500        # burn-in steps (discarded)
nsteps   = 3000       # production steps

# Initialise walkers in a small ball around the MAP estimate
p0_walkers = result_map.x + 1e-3 * rng.standard_normal((nwalkers, ndim))

sampler = emcee.EnsembleSampler(
    nwalkers, ndim, log_posterior, args=(x_data,)
)

# Burn-in phase
print("Running burn-in ...")
state = sampler.run_mcmc(p0_walkers, nburn, progress=True)
sampler.reset()

# Production phase
print("Running production chain ...")
sampler.run_mcmc(state, nsteps, progress=True)

print(f"\nAcceptance fraction (mean): {sampler.acceptance_fraction.mean():.3f}")

## 5. Trace plots

Trace plots show the chain values as a function of step number and are the primary diagnostic for convergence.  A well-mixed chain should look like white noise around the posterior mean.

In [ ]:
param_names = [r'$\theta$', r'$\mu$', r'$\sigma$', r'$\lambda$']
true_values = [theta_true, mu_true, sigma_true, lambda_true]

fig, axes = plt.subplots(ndim, figsize=(10, 8), sharex=True)
samples = sampler.get_chain()   # shape: (nsteps, nwalkers, ndim)
posterior_means = samples.mean(axis=(0, 1))

for i, ax in enumerate(axes):
    ax.plot(samples[:, :, i], alpha=0.3, lw=0.5, color='steelblue')
    ax.axhline(true_values[i], color='red', lw=1.5, ls='--', label='True value')
    ax.axhline(posterior_means[i], color='darkorange', lw=1.5, ls='-', label='Posterior mean')
    ax.set_ylabel(param_names[i])
    if i == 0:
        ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Step')
fig.suptitle('MCMC trace plots', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Posterior distributions and corner plot

Flatten the chain to obtain the posterior samples and visualise the full joint posterior with a **corner plot**. A posterior corner plot (or triangle plot) is a visualization tool used in Bayesian inference to display high-dimensional parameter distributions. It shows 1D marginalized posterior distributions for each parameter on the diagonal and 2D joint distributions (correlations) on the off-diagonal panels, highlighting 1σ, 2σ, and 3σ confidence intervals.

In [ ]:
from matplotlib.lines import Line2D

flat_samples = sampler.get_chain(flat=True)  # shape: (nsteps * nwalkers, ndim)
print(f"Total posterior samples: {flat_samples.shape[0]}")

sigma_levels = [1.0 - np.exp(-0.5 * n_sigma**2) for n_sigma in (1, 2, 3)]
contour_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig = corner.corner(
    flat_samples,
    labels=param_names,
    truths=true_values,
    truth_color='red',
    color='steelblue',
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt='.3f',
    title_kwargs={'fontsize': 10},
    plot_datapoints=False,
    fill_contours=False,
    levels=sigma_levels,
    contour_kwargs={'colors': contour_colors, 'linewidths': 1.5},
)

legend_handles = [
    Line2D([0], [0], color=contour_colors[0], lw=2, label='3σ contour (99.7%)'),
    Line2D([0], [0], color=contour_colors[1], lw=2, label='2σ contour (95.4%)'),
    Line2D([0], [0], color=contour_colors[2], lw=2, label='1σ contour (68.3%)'),
]
fig.legend(handles=legend_handles, loc='upper right', bbox_to_anchor=(0.98, 0.98), fontsize=9)

fig.suptitle('Joint posterior distribution', y=1.01, fontsize=13)
plt.show()

## 7. Marginal posterior for $\theta$

The **marginal posterior** for the parameter of interest $\theta$ is obtained by integrating (marginalising) over the nuisance parameters:

$$p(\theta \mid \mathbf{x}) = \int p(\theta,\mu,\sigma,\lambda \mid \mathbf{x})\, d\mu\, d\sigma\, d\lambda$$

In MCMC this is trivial: just look at the $\theta$ column of the flat samples.  The **credible interval** is read off directly from posterior quantiles.

In [ ]:
theta_samples = flat_samples[:, 0]

theta_median = np.median(theta_samples)
theta_lo68, theta_hi68 = np.percentile(theta_samples, [16, 84])
theta_lo95, theta_hi95 = np.percentile(theta_samples, [ 2.5, 97.5])

print("=== Marginal posterior for θ ===")
print(f"  Median        : {theta_median:.4f}")
print(f"  68 % credible interval: [{theta_lo68:.4f}, {theta_hi68:.4f}]")
print(f"  95 % credible interval: [{theta_lo95:.4f}, {theta_hi95:.4f}]")
print(f"  True value    : {theta_true}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(theta_samples, bins=60, density=True,
        histtype='stepfilled', alpha=0.4, color='steelblue',
        label='Marginal posterior')

ax.axvline(theta_median, color='steelblue', lw=2, ls='-',
           label=f'Posterior median = {theta_median:.3f}')
ax.axvline(theta_map, color='darkorange', lw=2, ls='--',
           label=f'MAP = {theta_map:.3f}')
ax.axvline(theta_true, color='red', lw=2, ls=':',
           label=f'True = {theta_true}')

ax.axvspan(theta_lo68, theta_hi68, alpha=0.15, color='steelblue',
           label='68 % credible interval')
ax.axvspan(theta_lo95, theta_hi95, alpha=0.08, color='steelblue',
           label='95 % credible interval')

ax.set_xlabel(r'$\theta$')
ax.set_ylabel('Posterior density')
ax.set_title(r'Marginal posterior for signal fraction $\theta$')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 8. Posterior predictive check

Draw a set of parameter vectors from the posterior and overlay the corresponding PDFs on the data histogram.  The spread of the lines visualises the **posterior uncertainty** on the PDF shape.

In [ ]:
xx = np.linspace(0, 14, 500)

idx = rng.integers(0, flat_samples.shape[0], size=200)
posterior_draws = flat_samples[idx]

pdf_map = (theta_map * norm.pdf(xx, mu_map, sigma_map)
           + (1 - theta_map) * expon.pdf(xx, scale=1.0/lam_map))

pdf_true_line = (theta_true * norm.pdf(xx, mu_true, sigma_true)
                 + (1 - theta_true) * expon.pdf(xx, scale=1.0/lambda_true))

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(x_data, bins=60, range=(0, 14), density=True,
        histtype='stepfilled', alpha=0.3, color='steelblue', label='Data')

for theta_s, mu_s, sigma_s, lam_s in posterior_draws:
    pdf_s = (theta_s * norm.pdf(xx, mu_s, sigma_s)
             + (1 - theta_s) * expon.pdf(xx, scale=1.0/lam_s))
    ax.plot(xx, pdf_s, color='steelblue', alpha=0.04, lw=1)

ax.plot(xx, pdf_true_line, 'k--', lw=2, label='True PDF')
ax.plot(xx, pdf_map, 'darkorange', lw=2, ls='-', label='MAP PDF')

# dummy line for legend
ax.plot([], [], color='steelblue', alpha=0.4, lw=2, label='Posterior draws (200)')

ax.set_xlabel('x')
ax.set_ylabel('Probability density')
ax.set_title('Posterior predictive check')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 9. Summary

The table below reports the fitted values from this run (MAP and posterior means).

In [ ]:
from IPython.display import Markdown, display

# Fallback in case posterior_means is not yet defined in the current kernel state.
if 'posterior_means' not in globals():
    posterior_means = sampler.get_chain().mean(axis=(0, 1))

map_values = [theta_map, mu_map, sigma_map, lam_map]
post_mean_values = posterior_means.tolist()

summary_md = (
    "| Parameter | True | MAP | Posterior mean | Posterior median | 68% CI |\n"
    "|---|---:|---:|---:|---:|---:|\n"
    f"| $\\theta$ | {theta_true:.4f} | {map_values[0]:.4f} | {post_mean_values[0]:.4f} | {theta_median:.4f} | [{theta_lo68:.4f}, {theta_hi68:.4f}] |\n"
    f"| $\\mu$ | {mu_true:.4f} | {map_values[1]:.4f} | {post_mean_values[1]:.4f} | — | — |\n"
    f"| $\\sigma$ | {sigma_true:.4f} | {map_values[2]:.4f} | {post_mean_values[2]:.4f} | — | — |\n"
    f"| $\\lambda$ | {lambda_true:.4f} | {map_values[3]:.4f} | {post_mean_values[3]:.4f} | — | — |"
)

display(Markdown(summary_md))

**Key observations:**
- The MAP estimate coincides with the MLE when uniform priors are used, as expected.
- MCMC provides the full joint posterior, from which any marginal distribution can be extracted by simply looking at one column of the samples.
- Marginalising over the nuisance parameters automatically propagates their uncertainty into the credible interval for $\theta$ — no profile likelihood calculation is required.
- The posterior predictive check shows that the range of PDFs consistent with the data envelopes the true PDF well.

## Exercises

1. **Prior sensitivity.** Replace the uniform prior on $\theta$ with a $\text{Beta}(2,5)$ prior (encoding a belief that signal fractions above ~0.5 are unlikely). How does the marginal posterior for $\theta$ change? What happens as $n \to \infty$?

2. **Informative prior on $\mu$.** Suppose a previous experiment measured the signal peak position: $\mu_{\rm prev} = 5.1 \pm 0.2$. Encode this as a Gaussian prior $\pi(\mu) = \mathcal{N}(5.1, 0.2^2)$ and study how the posterior on $\mu$ and the marginal posterior on $\theta$ change.

3. **MCMC diagnostics.** Compute the **integrated autocorrelation time** $\tau$ for each parameter using `emcee`'s built-in `get_autocorr_time()`. Verify that the chain is long enough ($n_{\rm steps} \gg 50\,\tau$). If not, how many steps are needed?

4. **Credible vs. confidence intervals.** Compare the 95 % Bayesian credible interval for $\theta$ with the 95 % frequentist confidence interval from notebook 02. Do they agree? Under what conditions are they numerically equivalent?

5. **Marginalising a nuisance parameter analytically.** For the simpler case where $\mu$ and $\sigma$ are known (fixed to truth) and only $\theta$ and $\lambda$ are free, write down the posterior $p(\theta, \lambda \mid \mathbf{x})$ with uniform priors and sketch the marginal $p(\theta \mid \mathbf{x})$. Can you evaluate the marginalisation integral numerically on a 2-D grid (without MCMC)?